# Colab 01 - Build Embeddings / Qdrant Index

Run this notebook on Colab GPU to build the real Qdrant index with BioMedBERT and BioCLIP.

Required Colab Secrets:
- `QDRANT_URL`
- `QDRANT_API_KEY`

Optional:
- `OPENROUTER_API_KEY` for later generation/evaluation


In [ ]:
REPO_URL = "https://github.com/phamdinhhai/project-ks2.git"
PROJECT_DIR = "/content/project-ks2"

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only


In [ ]:
!python -m pip install -U pip
!pip install -e ".[gpu,qdrant,agent,eval]"
!pip install requests accelerate bitsandbytes qwen-vl-utils


In [ ]:
# Configure secrets from Colab Secrets. Do not hardcode keys in notebook.
import os
try:
    from google.colab import userdata
    for name in ["OPENROUTER_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]:
        value = userdata.get(name)
        if value:
            os.environ[name] = value
except Exception as exc:
    print("Colab userdata unavailable:", exc)

os.environ.setdefault("OPENROUTER_MODEL", "google/gemini-2.5-flash")
os.environ.setdefault("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")

print("OPENROUTER_API_KEY set:", bool(os.environ.get("OPENROUTER_API_KEY")))
print("OPENROUTER_MODEL:", os.environ.get("OPENROUTER_MODEL"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY set:", bool(os.environ.get("QDRANT_API_KEY")))


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


In [ ]:
# Verify Qdrant Cloud auth before long indexing
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth


In [ ]:
# Optional model smoke test. If this is too slow, skip and go directly to indexing.
!python -m medical_rag test-encoders --no-mock --include-bge


In [ ]:
# Build real index. Start small; increase --limit after first successful run.
!python -m medical_rag build-qdrant-index   --data-dir data   --qdrant-url "$QDRANT_URL"   --use-cloud-auth   --limit 1000   --recreate   --no-use-mock-models


In [ ]:
# Verify point counts after indexing
!python -m medical_rag test-qdrant --qdrant-url "$QDRANT_URL" --use-cloud-auth
